# 4 — High-gap posts + Embedding-based Mental-Health Flags

This notebook uses **cosine similarity** to a small set of **mental-health seed sentences** to create:

- `mh_sim_max`: maximum cosine similarity of a post to any seed sentence
- `mh_flag`: a binary flag based on either a **quantile** threshold or a **fixed** cutoff
- `mh_best_seed`: which seed sentence matched best (and the similarity score)

## Inputs
- `data/processed/3-gap_and_swear_features.csv` (from Notebook 3)

## Outputs
- `data/processed/writestreak_posts_highgap_with_mh_embedding.csv` (high-gap subset with mh signals)
- optional: `data/processed/writestreak_posts_all_with_mh_embedding.csv` (if "all posts" section enabled)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
processed_dir = REPO_ROOT / ".." / "data" / "processed"

notebook3DataPath = processed_dir / "3-gap_and_swear_features.csv"
df = pd.read_csv(notebook3DataPath)

print("Loaded:", notebook3DataPath)
print("Rows:", len(df), "Users:", df["user_id"].nunique())
df.head(2)


Loaded: c:\Users\hus44\Code\Directed-Reading-Project\notebooks\..\data\processed\3-gap_and_swear_features.csv
Rows: 16471 Users: 869


,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,...,high_gap_gapmag_mean,high_gap_gapmag_max,high_gap_valence_mean_signed,high_gap_arousal_mean_signed,swear_count,swear_density,swear_present,swear_unique_count,swear_types,high_gap_swear_count
0,kt7xl7,t3_kt7xl7,Namatl,00052db50867a4da,1.610129e+09,2021-01-08 18:04:23+00:00,Streak 1 Writing Prompt,"""I was walking to the store in the evening, th...",WriteStreakEN,/r/WriteStreakEN/comments/kt7xl7/streak_1_writ...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
1,ku0tf4,t3_ku0tf4,Namatl,00052db50867a4da,1.610231e+09,2021-01-09 22:18:54+00:00,Streak 2 Favorite Singer/Band,Mi favorite singer is Michael. I grew up liste...,WriteStreakEN,/r/WriteStreakEN/comments/ku0tf4/streak_2_favo...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0


### Choosing high gap subset

In [2]:
df_hg = df[df["high_gap_count"].fillna(0) > 0].copy()
df_non = df[df["high_gap_count"].fillna(0) == 0].copy()

print("High-gap posts:", len(df_hg))
print("Non-high-gap posts:", len(df_non))

High-gap posts: 9168
Non-high-gap posts: 7303


### Embedding model, seed sentences, and cosine similarity

We compute cosine similarities between each post and each seed, then store:

- `mh_sim_max`: max similarity across seeds
- `mh_sim_mean_top3`: mean of top 3 similarities (optional robustness)
- `mh_best_seed_idx`, `mh_best_seed`, `mh_best_seed_sim`: interpretable "closest seed"

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

SEEDS = [
    "I feel anxious or worried and I can't stop thinking about it.",
    "I feel sad, hopeless, or like nothing will get better.",
    "I feel overwhelmed, stressed out, or burned out.",
    "I feel lonely or isolated and like I have no support.",
    "I have trouble sleeping or I am exhausted all the time.",
    "I feel guilty, ashamed, or like I am a failure.",
    "I feel emotionally numb or empty.",
    "I am struggling to cope with my feelings or daily life."
]

texts = df_hg["text"].fillna("").astype(str).tolist()

seed_emb = model.encode(SEEDS, normalize_embeddings=True)
post_emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

sims = cosine_similarity(post_emb, seed_emb)

df_hg["mh_sim_max"] = sims.max(axis=1)
df_hg["mh_sim_mean_top3"] = np.sort(sims, axis=1)[:, -3:].mean(axis=1)

best_idx = sims.argmax(axis=1)
df_hg["mh_best_seed_idx"] = best_idx
df_hg["mh_best_seed"] = [SEEDS[i] for i in best_idx]
df_hg["mh_best_seed_sim"] = sims[np.arange(len(df_hg)), best_idx]

df_hg[["post_id","mh_sim_max","mh_best_seed_sim","mh_best_seed_idx"]].head()

c:\Users\hus44\AppData\Local\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7438.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 287/287 [02:53<00:00,  1.65it/s]


,post_id,mh_sim_max,mh_best_seed_sim,mh_best_seed_idx
3,j4p7n0,0.126164,0.126164,0
5,kk1866,0.359103,0.359103,2
6,18s52qq,0.197604,0.197604,5
7,18sthvq,0.226495,0.226495,6
8,18tq0cp,0.132169,0.132169,1


### Create `mh_flag`

Two options:

- `quantile`: flags the top *q* fraction of posts as mental-health-relevant 
- `fixed`: uses a fixed similarity cutoff


In [4]:
MH_MODE = "quantile"  # "quantile" or "fixed"

if MH_MODE == "quantile":
    q = 0.90
    thr = float(df_hg["mh_sim_max"].quantile(q))
    df_hg["mh_flag"] = (df_hg["mh_sim_max"] >= thr).astype(int)
    print("Using quantile threshold:", q, "->", thr)
else:
    thr = 0.45
    df_hg["mh_flag"] = (df_hg["mh_sim_max"] >= thr).astype(int)
    print("Using fixed threshold:", thr)

df_hg["mh_flag"].value_counts(dropna=False)

flagged = df_hg[df_hg["mh_flag"] == 1].copy()

seed_summary = (
    flagged.groupby(["mh_best_seed_idx","mh_best_seed"])
    .agg(
        n_posts=("post_id","count"),
        mean_best_sim=("mh_best_seed_sim","mean"),
        mean_max_sim=("mh_sim_max","mean"),
    )
    .reset_index()
    .sort_values(["n_posts","mean_best_sim"], ascending=[False, False])
)

seed_summary

cols = ["post_id","title","mh_sim_max","mh_best_seed_sim","mh_best_seed","text"]
flagged.sort_values("mh_sim_max", ascending=False)[cols].head(10)

Using quantile threshold: 0.9 -> 0.3476814448833466


,post_id,title,mh_sim_max,mh_best_seed_sim,mh_best_seed,text
6291,1gmf91v,Streak 4. Let’s talk about loneliness,0.663514,0.663514,I feel lonely or isolated and like I have no s...,Streak 4. Let’s talk about loneliness\n\nHi bo...
12543,rlyy85,Streak 4: Being alone vs loneliness,0.629758,0.629758,I feel lonely or isolated and like I have no s...,Streak 4: Being alone vs loneliness\n\nI reall...
14401,1l70k3r,Streak 5 - sleep,0.603101,0.603101,I have trouble sleeping or I am exhausted all ...,Streak 5 - sleep\n\nI don't know why I am so t...
12593,v9rpz0,Streak 31' Sleeping,0.602139,0.602139,I have trouble sleeping or I am exhausted all ...,Streak 31' Sleeping\n\nI haven't slept much so...
3949,w34mb1,Streak 375: alone not lonely.,0.600516,0.600516,I feel lonely or isolated and like I have no s...,Streak 375: alone not lonely.\n\nI've spent th...
14374,1jcnyem,Streak 45 - tired,0.590225,0.590225,I have trouble sleeping or I am exhausted all ...,Streak 45 - tired\n\nI'm so tired. Almost the ...
5874,qd5xu4,Streak 12: A Blue Day,0.589718,0.589718,I feel lonely or isolated and like I have no s...,Streak 12: A Blue Day\n\nI felt blue before. T...
6983,sy2z1d,streak 130 - can’t tolerate sleep deprivation,0.586726,0.586726,I have trouble sleeping or I am exhausted all ...,streak 130 - can’t tolerate sleep deprivation\...
9163,lam40h,Streak 19: Sleep well,0.571949,0.571949,I have trouble sleeping or I am exhausted all ...,Streak 19: Sleep well\n\nSleep is an important...
12387,pon5zc,Streak 2: Learning vocabularies and my problems .,0.569603,0.569603,"I feel overwhelmed, stressed out, or burned out.",Streak 2: Learning vocabularies and my problem...


In [6]:
DO_NON_HIGH_GAP = False

if DO_NON_HIGH_GAP:
    texts_non = df_non["text"].fillna("").astype(str).tolist()
    post_emb_non = model.encode(texts_non, normalize_embeddings=True, show_progress_bar=True)
    sims_non = cosine_similarity(post_emb_non, seed_emb)
    df_non["mh_sim_max"] = sims_non.max(axis=1)
    print("Mean mh_sim_max (high-gap):", df_hg["mh_sim_max"].mean())
    print("Mean mh_sim_max (non-high-gap):", df_non["mh_sim_max"].mean())
    
out_hg = processed_dir / "4-writestreak_posts_highgap_with_mh_embedding.csv"
df_hg.to_csv(out_hg, index=False)
print("Saved:", out_hg)

Saved: c:\Users\hus44\Code\Directed-Reading-Project\notebooks\..\data\processed\4-writestreak_posts_highgap_with_mh_embedding.csv
